<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from tensorflow.keras.datasets import cifar10

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("assignment")

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
def load_data(seed):
    (X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

    print("Formato original do X_train:", X_train_full.shape)

    y_train_full = y_train_full.ravel()

    X = X_train_full.reshape(X_train_full.shape[0], -1)
    print("Formato após flatten:", X.shape)

    X = X.astype(np.float32) / 255.0

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y_train_full,
        test_size=0.2,
        random_state=seed,
        stratify=y_train_full
    )

    return X_train, X_val, y_train, y_val

O formato original das imagens do CIFAR-10 é `(32, 32, 3)`, pois cada imagem possui 32 pixels de altura, 32 pixels de largura e 3 canais de cor RGB.

Após o flatten, cada imagem passa a ter 3072 features, pois `32 × 32 × 3 = 3072`.

O flatten é necessário porque uma MLP recebe um vetor unidimensional como entrada. Como as imagens originalmente estão em formato tridimensional, elas precisam ser convertidas para vetores antes do treinamento.

A normalização é importante porque os valores dos pixels originalmente variam de 0 a 255. Ao dividir por 255, os valores passam para o intervalo entre 0 e 1, o que melhora a estabilidade do treinamento e facilita a convergência do modelo.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=8,
    batch_size=256
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=seed,
        solver="adam",
        shuffle=True,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=3,
        tol=1e-3
    )

    model.fit(X_train, y_train)

    return model


def count_first_layer_params(input_features, hidden_layers):
    first_hidden_layer = hidden_layers[0]
    return (input_features + 1) * first_hidden_layer


count_first_layer_params(3072, (64,))

O número de parâmetros da primeira camada depende da quantidade de neurônios escolhida. Como cada imagem possui 3072 features após o flatten, a primeira camada possui `(3072 + 1) × número de neurônios`, considerando também o bias. Por exemplo, com 64 neurônios, a primeira camada possui `(3072 + 1) × 64 = 196672` parâmetros.

A função ReLU introduz não linearidade na rede. Ela retorna zero para valores negativos e retorna o próprio valor quando a entrada é positiva. Essa função é muito usada porque é simples, eficiente e ajuda a reduzir problemas de saturação durante o treinamento.

MLPs possuem muitos parâmetros ao trabalhar com imagens porque cada pixel passa a ser tratado como uma feature independente. Como imagens possuem muitos pixels e canais de cor, o número de conexões entre a entrada e as camadas ocultas cresce rapidamente.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    }

    return metrics

A função `evaluate` realiza as predições do modelo e calcula as principais métricas de classificação.

A accuracy representa a proporção total de previsões corretas feitas pelo modelo.

A precision indica, entre as amostras previstas como uma determinada classe, quantas realmente pertenciam a essa classe. Já o recall indica, entre as amostras que realmente pertenciam a uma classe, quantas foram corretamente identificadas pelo modelo.

O f1-score é importante quando queremos equilibrar precision e recall. Ele é especialmente útil quando a accuracy sozinha não é suficiente para interpretar o desempenho do modelo. Como o CIFAR-10 é um problema multiclasse, foi utilizada a média `weighted`, considerando a proporção de amostras de cada classe.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
seed = 42

X_train, X_val, y_train, y_val = load_data(seed)

# Os experimentos usam uma subamostra estratificada para manter o tempo de execução viável
# no Colab e no GitHub Actions, já que MLPs em imagens RGB possuem muitos parâmetros.
def select_stratified_subset(X, y, n_samples, seed):
    if n_samples >= len(X):
        return X, y

    X_subset, _, y_subset, _ = train_test_split(
        X,
        y,
        train_size=n_samples,
        random_state=seed,
        stratify=y
    )

    return X_subset, y_subset


X_train_exp, y_train_exp = select_stratified_subset(X_train, y_train, 6000, seed)
X_val_exp, y_val_exp = select_stratified_subset(X_val, y_val, 2000, seed)

print("Treino usado nos experimentos:", X_train_exp.shape)
print("Validação usada nos experimentos:", X_val_exp.shape)

all_results = []

In [ ]:
def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=8,
    batch_size=256,
    run_name=None
):
    if run_name is None:
        run_name = f"act={activation}_layers={hidden_layers}_lr={learning_rate}"

    if mlflow.active_run():
        mlflow.end_run()

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("seed", seed)

        start_time = time.time()

        try:
            model = train_mlp(
                X_train=X_train,
                y_train=y_train,
                activation=activation,
                hidden_layers=hidden_layers,
                learning_rate=learning_rate,
                seed=seed,
                max_iter=max_iter,
                batch_size=batch_size
            )

            training_time = time.time() - start_time
            metrics = evaluate(model, X_val, y_val)

            final_loss = model.loss_ if hasattr(model, "loss_") else None
            n_iter = model.n_iter_ if hasattr(model, "n_iter_") else None
            status = "success"
            error_message = ""

        except Exception as error:
            model = None
            training_time = time.time() - start_time
            metrics = {
                "accuracy": 0.0,
                "precision": 0.0,
                "recall": 0.0,
                "f1_score": 0.0
            }
            final_loss = None
            n_iter = None
            status = "failed"
            error_message = f"{type(error).__name__}: {error}"
            mlflow.log_param("error_type", type(error).__name__)

        mlflow.log_param("status", status)
        mlflow.log_metric("accuracy", metrics["accuracy"])
        mlflow.log_metric("precision", metrics["precision"])
        mlflow.log_metric("recall", metrics["recall"])
        mlflow.log_metric("f1_score", metrics["f1_score"])
        mlflow.log_metric("training_time", training_time)

        if final_loss is not None:
            mlflow.log_metric("final_loss", final_loss)

        if n_iter is not None:
            mlflow.log_metric("n_iter", n_iter)

        result = {
            "activation": activation,
            "hidden_layers": hidden_layers,
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "accuracy": metrics["accuracy"],
            "precision": metrics["precision"],
            "recall": metrics["recall"],
            "f1_score": metrics["f1_score"],
            "training_time": training_time,
            "final_loss": final_loss,
            "n_iter": n_iter,
            "status": status,
            "error_message": error_message
        }

    return model, result

In [ ]:
baseline_model, baseline_result = run_experiment(
    X_train=X_train_exp,
    y_train=y_train_exp,
    X_val=X_val_exp,
    y_val=y_val_exp,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    seed=seed,
    max_iter=8,
    batch_size=256,
    run_name="baseline_relu_64_lr_0001"
)

all_results.append(baseline_result)

pd.DataFrame(all_results)

O experimento com melhor desempenho deve ser identificado pela comparação das métricas registradas, principalmente accuracy e f1-score. A configuração mais estável tende a ser aquela que combina bom desempenho, menor loss final e tempo de treinamento aceitável.

O benefício do rastreamento experimental com MLflow é permitir comparar diferentes configurações de forma organizada e reprodutível. Com ele, é possível registrar hiperparâmetros, métricas e tempo de treinamento, facilitando a análise crítica dos resultados.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
activations = ["logistic", "tanh", "relu"]

activation_results = []

for activation in activations:
    model, result = run_experiment(
        X_train=X_train_exp,
        y_train=y_train_exp,
        X_val=X_val_exp,
        y_val=y_val_exp,
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
        seed=seed,
        max_iter=8,
        batch_size=256,
        run_name=f"activation_{activation}"
    )

    activation_results.append(result)
    all_results.append(result)

activation_results_df = pd.DataFrame(activation_results)
activation_results_df.sort_values(by="f1_score", ascending=False)

In [ ]:
best_activation = activation_results_df.sort_values(by="f1_score", ascending=False).iloc[0]

print("Melhor ativação por f1-score:", best_activation["activation"])
print("Accuracy:", best_activation["accuracy"])
print("F1-score:", best_activation["f1_score"])
print("Loss final:", best_activation["final_loss"])

A ativação com melhor desempenho deve ser observada na tabela e no resumo acima, considerando principalmente o f1-score e a accuracy.

A convergência e a estabilidade podem ser analisadas pela combinação entre bom desempenho, menor loss final e número de iterações. Em geral, a ReLU costuma apresentar boa convergência em redes neurais, pois reduz problemas de saturação e facilita o fluxo do gradiente. Porém, o resultado final pode variar de acordo com a arquitetura, o learning rate e a amostra utilizada.

Houve diferenças no treinamento entre logistic, tanh e ReLU, principalmente em relação à loss final, ao tempo de treinamento e às métricas finais.

A ReLU é amplamente utilizada em Deep Learning porque é simples, computacionalmente eficiente e ajuda a reduzir o problema do desaparecimento do gradiente, comum em funções como logistic e tanh.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
architectures = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

architecture_results = []

for hidden_layers in architectures:
    model, result = run_experiment(
        X_train=X_train_exp,
        y_train=y_train_exp,
        X_val=X_val_exp,
        y_val=y_val_exp,
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
        seed=seed,
        max_iter=8,
        batch_size=256,
        run_name=f"architecture_{hidden_layers}"
    )

    architecture_results.append(result)
    all_results.append(result)

architecture_results_df = pd.DataFrame(architecture_results)
architecture_results_df.sort_values(by="f1_score", ascending=False)

In [ ]:
best_architecture = architecture_results_df.sort_values(by="f1_score", ascending=False).iloc[0]
most_expensive_architecture = architecture_results_df.sort_values(by="training_time", ascending=False).iloc[0]

print("Melhor arquitetura por f1-score:", best_architecture["hidden_layers"])
print("F1-score:", best_architecture["f1_score"])
print("Arquitetura com maior custo computacional:", most_expensive_architecture["hidden_layers"])
print("Tempo de treinamento:", most_expensive_architecture["training_time"])

Redes maiores não necessariamente melhoram os resultados. Arquiteturas maiores possuem mais capacidade de aprendizado, mas também aumentam o custo computacional e podem apresentar maior risco de overfitting.

A arquitetura com melhor tradeoff é a que combina bom f1-score, boa accuracy e tempo de treinamento razoável. Caso uma arquitetura maior apresente desempenho muito próximo de uma menor, mas com tempo de treinamento superior, a menor pode ser considerada uma escolha melhor.

Sinais de overfitting podem ser suspeitados quando uma arquitetura muito grande aumenta o custo computacional sem ganho relevante nas métricas de validação. Para confirmar overfitting com mais segurança, seria necessário comparar também o desempenho no conjunto de treino.

A arquitetura com maior custo computacional deve ser observada pelo maior `training_time` na tabela acima.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rates = [0.1, 0.01, 0.001]

learning_rate_results = []

for lr in learning_rates:
    model, result = run_experiment(
        X_train=X_train_exp,
        y_train=y_train_exp,
        X_val=X_val_exp,
        y_val=y_val_exp,
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        seed=seed,
        max_iter=8,
        batch_size=256,
        run_name=f"learning_rate_{lr}"
    )

    learning_rate_results.append(result)
    all_results.append(result)

learning_rate_results_df = pd.DataFrame(learning_rate_results)
learning_rate_results_df.sort_values(by="f1_score", ascending=False)

In [ ]:
best_lr = learning_rate_results_df.sort_values(by="f1_score", ascending=False).iloc[0]
worst_lr = learning_rate_results_df.sort_values(by="f1_score", ascending=True).iloc[0]

print("Melhor learning rate por f1-score:", best_lr["learning_rate"])
print("F1-score:", best_lr["f1_score"])
print("Learning rate com pior desempenho/maior instabilidade:", worst_lr["learning_rate"])
print("F1-score:", worst_lr["f1_score"])

O learning rate com melhor desempenho deve ser identificado pela tabela e pelo resumo acima, considerando accuracy, f1-score e loss final.

Um learning rate muito alto, como 0.1, pode tornar o treinamento instável, pois os pesos são atualizados com passos muito grandes. Isso pode fazer com que o modelo tenha dificuldade de convergir ou apresente queda nas métricas.

Um learning rate muito baixo pode tornar o treinamento mais estável, mas também mais lento, pois as atualizações nos pesos são pequenas.

Em geral, valores menores, como 0.001, tendem a apresentar maior estabilidade em MLPs treinadas com Adam, enquanto valores maiores podem causar maior instabilidade.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
all_results_df = pd.DataFrame(all_results)
all_results_df.sort_values(by="f1_score", ascending=False)

In [ ]:
best_overall = all_results_df.sort_values(by="f1_score", ascending=False).iloc[0]

print("Melhor configuração final:")
print("Ativação:", best_overall["activation"])
print("Arquitetura:", best_overall["hidden_layers"])
print("Learning rate:", best_overall["learning_rate"])
print("Accuracy:", best_overall["accuracy"])
print("F1-score:", best_overall["f1_score"])
print("Loss final:", best_overall["final_loss"])
print("Tempo de treinamento:", best_overall["training_time"])

Com base nos experimentos realizados, foi possível observar que o comportamento da loss e das métricas foi influenciado principalmente pelo learning rate, pela arquitetura da rede e pela função de ativação.

O learning rate teve impacto direto na estabilidade do treinamento. Valores muito altos podem gerar instabilidade e dificultar a convergência, enquanto valores menores tendem a tornar o treinamento mais controlado.

A arquitetura também influenciou os resultados. Redes muito pequenas podem ter baixa capacidade de aprendizado, enquanto redes maiores aumentam o número de parâmetros, o tempo de treinamento e o risco de overfitting. Por isso, a melhor arquitetura não é necessariamente a maior, mas aquela que apresenta melhor equilíbrio entre desempenho e custo computacional.

As funções de ativação também afetaram o treinamento. A ReLU é geralmente eficiente em redes neurais porque facilita o fluxo do gradiente e reduz problemas de saturação. Já logistic e tanh podem apresentar maior dificuldade dependendo da configuração experimental.

A melhor configuração final é a exibida no resumo acima, escolhida pelo maior f1-score entre os experimentos realizados.

As principais dificuldades observadas foram o tempo de treinamento dos modelos, a sensibilidade ao learning rate e o grande número de parâmetros gerado pelo uso de MLPs em imagens.

MLPs possuem limitações para imagens porque o flatten remove a estrutura espacial da imagem. Assim, relações importantes entre pixels vizinhos podem ser perdidas. Modelos convolucionais, como CNNs, são mais adequados para imagens porque conseguem explorar padrões espaciais locais.

O backpropagation contribui para o aprendizado ao calcular os gradientes do erro em relação aos pesos da rede. Com esses gradientes, o otimizador ajusta os pesos de forma iterativa, reduzindo a loss e melhorando o desempenho do modelo ao longo do treinamento.